Importing modules

In [38]:
import numpy as np

Defining users and channels

In [39]:
n_users = 35      # secondary users
n_channels = 30     # available spectrum bands

Getting SINR from random

In [40]:
# Simulate channel gains randomly (in reality this comes from pathloss model)
# SINR[i][j] = signal quality if user i uses channel j
np.random.seed(67)
channel_gain = np.random.uniform(0.1, 1.0, (n_users, n_channels)) # to get SINR which gets throughput
noise = 0.1

SINR = channel_gain / noise  # simplified, no inter-user interference yet

SINR

array([[5.91266287, 8.7297095 , 7.17303328, ..., 7.37628066, 9.58082098,
        6.41732327],
       [4.62026413, 4.1943718 , 8.32747141, ..., 3.15823553, 7.08705175,
        4.86264092],
       [3.02554615, 2.36651987, 5.08196692, ..., 5.74216793, 3.91899364,
        1.88450753],
       ...,
       [5.15590223, 2.87792417, 2.39532817, ..., 5.85425138, 8.14644798,
        7.19077096],
       [7.025241  , 4.20727915, 4.47077576, ..., 8.65510371, 7.61359485,
        3.77042225],
       [9.67435321, 1.47319415, 7.06419683, ..., 7.24148268, 7.59506263,
        2.86737154]], shape=(35, 30))

Setting up Primary Users

In [41]:
n = n_channels          # array size
k = 5                   # number of busy channels

arr = np.zeros(n, dtype=int)
arr[np.random.choice(n, k, replace=False)] = 1

# Primary user occupancy: PU[j] = 1 means channel j is occupied by primary user
PU_occupied = arr 
PU_occupied

array([0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 1])

Fitness function which gets the throughput using SINR and adds penalty

In [42]:
def fitness(x):
    channels = np.clip(np.round(x).astype(int), 0, n_channels - 1)
    
    throughput = 0
    penalty = 0
    for user in range(n_users):
        ch = channels[user]
        if PU_occupied[ch]:
            penalty += 1000  # hard penalty for interfering with primary user
        else:
            throughput += np.log2(1 + SINR[user, ch])
    
    # secondary user collision penalty
    for ch in range(n_channels):
        users_on_ch = np.sum(channels == ch)
        if users_on_ch > 1:
            penalty += (users_on_ch - 1) * 80
    
    return -throughput + penalty




PSO algorithm

In [ ]:
def dpso(f, D, N, S, n_iter, a, b, b_hat, c):

    x = np.random.uniform(0, S, size=(N, D))                 # setting up the initial positions of the N number of particles
    v = np.random.normal(size=(N, D))                        # setting up the initial velocities
    p = x.copy()                                             # best particle position
    fp = np.array([f(p[i]) for i in range(N)])               # throughput of all particles
    p_hat = x[np.argmin(fp)].copy()                          # global best position
    fp_hat = f(p_hat)                                        # throughput of global best position

    for i in range (n_iter):

        if i%(n_iter//10)== 0:                               # to show progress
            print(f"progress {(i/n_iter)*100:.0f}%")

        r,r_hat = np.random.uniform(0, 1, (2,N, D))          # setting up random parameters

        v = a*v + b*r*(p-x) + b_hat*r_hat*(p_hat-x)          # updating velocities as vector sum of the directions of initial velocity, local minima, local maxima
        x = x + c*v                                          # updating position according to velocities

        for n in range(N):                                   # calculation for each particle

            xn = x[n]                                        # getting position of that particle           
            fxn = f(xn)                                      # current throughput of that particle
            fpn = fp[n]                                      # best throughput of that particle

            if fxn < fpn:                                    # if current throughput of that particle is better the previous ones, update
                p[n] = xn.copy()
                fp[n] = fxn

                if fxn < fp_hat:                             # if the current througput is the global best throughput, update
                    p_hat = xn.copy()
                    fp_hat = fxn
    
    print("progress 100%")
    return p_hat                                             # "coordinates", ie channel allocation of global best throughput

        



Calling PSO and printing throughput

In [44]:
result = dpso(
    f=fitness,
    N=35,           # particles
    S=n_channels-1, # search space [0, n_channels-1]
    a=0.7,
    b=1.5,
    b_hat=1.5,
    c=0.1,
    D=n_users,       # dimensions = number of users
    n_iter = 500
)


best_assignment = np.clip(np.round(result).astype(int), 0, n_channels-1)

actual_throughput = 0
for user in range(n_users):
    ch = best_assignment[user]
    if not PU_occupied[ch]:
        actual_throughput += np.log2(1 + SINR[user, ch])

print("Channel assignment:", best_assignment)
print("Throughput:", actual_throughput)

progress 0%
progress 10%
progress 20%
progress 30%
progress 40%
progress 50%
progress 60%
progress 70%
progress 80%
progress 90%
progress 100%
Channel assignment: [23 22 19 28 17 27 14 28 23 19  0 15 16  2  2 15  1 26 19  3 10  7 26 17
  5 20  5  2 16 21  9 13 18 21  7]
Throughput: 90.77135031880624


In [45]:
def qpso(f, N, beta_start, beta_end, n_iter, D, low, high):
    
    # Initialize positions
    x = np.random.uniform(low, high, size=(N, D))
    
    # Personal bests
    p = x.copy()
    p_fitness = np.array([f(p[i]) for i in range(N)])
    
    # Global best
    g = p[np.argmin(p_fitness)]
    
    for t in range(n_iter):
        if t%100 == 0:
                print("doing...",t)
        
        # Beta decreases linearly from beta_start to beta_end
        beta = beta_start - (beta_start - beta_end) * t / n_iter
        
        # Mean best position — average of all personal bests
        mbest = np.mean(p, axis=0)
        
        for i in range(N):
            
            # Local attractor — random point between personal best and global best
            phi = np.random.uniform(0, 1, D)
            p_local = phi * p[i] + (1 - phi) * g
            
            # Sample new position from Laplace distribution
            u = np.random.uniform(0, 1, D)
            sign = np.where(np.random.rand(D) < 0.5, 1, -1)
            x[i] = p_local + sign * beta * np.abs(mbest - x[i]) * np.log(1/u)
            
            # Clip to valid range
            x[i] = np.clip(x[i], low, high)
            
            # Update personal best
            if f(x[i]) < p_fitness[i]:
                p[i] = x[i].copy()
                p_fitness[i] = f(x[i])
            
            # Update global best
            if p_fitness[i] < f(g):
                g = p[i].copy()
    
    return g




Calling QPSO and printing throughput

In [46]:
result_qpso = qpso(
    f=fitness,
    N=35,
    beta_start=1.0,
    beta_end=0.5,
    n_iter=500,
    D=n_users,
    low=0,
    high=n_channels-1
)

best_assignment = np.clip(np.round(result_qpso).astype(int), 0, n_channels-1)
actual_throughput = 0
for user in range(n_users):
    ch = best_assignment[user]
    if not PU_occupied[ch]:
        actual_throughput += np.log2(1 + SINR[user, ch])

print("Channel assignment:", best_assignment)
print("Throughput:", actual_throughput)

doing... 0
doing... 100
doing... 200
doing... 300
doing... 400
Channel assignment: [ 9 23 22 19  5  0  1 13 16 11 20 17 14  1 21  9 14  7  3 21 15 27 17 26
 14 13  3 28 17 18  2 21 19 23 10]
Throughput: 90.58509993953368
